# IdleSparkle Waveform Tuner

Interactive sliders for the additive-synthesis brightness waveform.  
Drag a slider → graph updates on release.

When you've found values you like, run the **Export** cell at the bottom to print the constants ready to paste into `lights.py`.

In [ ]:
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
from IPython.display import clear_output

In [ ]:
# ── Base parameters (fixed detuning structure — mirrors lights.py) ────────────

_TAU = 2.0 * math.pi

# (phi, T1, T2, T3, T_med, t_off_med, T_slow, t_off_slow)
_PIXELS_BASE = (
    (0.0,  5.1,  7.7, 11.9,  67.0,  40.2,  331.0, 198.6),
    (2.1,  5.3,  8.1, 12.7,  71.0,  49.7,  349.0, 244.3),
    (4.2,  4.9,  7.3, 11.3,  61.0,  48.8,  313.0, 250.4),
)

def _make_pixels(med_period, slow_period):
    """Scale per-pixel periods around new centre values, preserving relative detuning."""
    out = []
    for phi, T1, T2, T3, Tm, t_off_m, Ts, t_off_s in _PIXELS_BASE:
        Tm_new    = med_period  * (Tm / 67.0)
        t_off_m   = Tm_new * 0.6
        Ts_new    = slow_period * (Ts / 331.0)
        t_off_s   = Ts_new * 0.6
        out.append((phi, T1, T2, T3, Tm_new, t_off_m, Ts_new, t_off_s))
    return out


def _brightness(t, phi, T1, T2, T3, Tm, t_off_m, Ts, t_off_s,
                dc, A1, A2, A3, amed, aslo, pow_):
    b  = dc
    b += A1 * math.sin(_TAU * t / T1 + phi)
    b += A2 * math.sin(_TAU * t / T2 + phi * 1.3)
    b += A3 * math.sin(_TAU * t / T3 + phi * 0.7)
    sm = max(0.0, math.sin(_TAU * (t + t_off_m) / Tm))
    b += amed * sm ** pow_
    ss = max(0.0, math.sin(_TAU * (t + t_off_s) / Ts))
    b += aslo * ss ** pow_
    return max(0.0, min(1.0, b))


def _contributions(t, phi, T1, T2, T3, Tm, t_off_m, Ts, t_off_s,
                   dc, A1, A2, A3, amed, aslo, pow_):
    fast = (dc
            + A1 * math.sin(_TAU * t / T1 + phi)
            + A2 * math.sin(_TAU * t / T2 + phi * 1.3)
            + A3 * math.sin(_TAU * t / T3 + phi * 0.7))
    sm   = max(0.0, math.sin(_TAU * (t + t_off_m) / Tm))
    ss   = max(0.0, math.sin(_TAU * (t + t_off_s) / Ts))
    return fast, amed * sm ** pow_, aslo * ss ** pow_


def _ewma(arr, alpha):
    out = np.empty_like(arr)
    s = 0.0
    for j, v in enumerate(arr):
        s = alpha * v + (1.0 - alpha) * s
        out[j] = s
    return out

In [ ]:
# ── Interactive plot ──────────────────────────────────────────────────────────

_PX_COLORS = ["#d95f02", "#1b9e77", "#7570b3"]
_REF_LINES = [
    (0.9, "--", 0.35, "#999", "0.90 rare peak"),
    (0.6, "--", 0.45, "#999", "0.60 medium peak"),
    (0.2, ":",  0.55, "#aaa", "0.20 floor"),
]


def _plot(smooth, pow_, amed, aslo, dc, med_period, slow_period, no_texture, duration_min):
    clear_output(wait=True)

    A1 = 0.0 if no_texture else 0.05
    A2 = 0.0 if no_texture else 0.04
    A3 = 0.0 if no_texture else 0.03
    wave_kw = dict(dc=dc, A1=A1, A2=A2, A3=A3, amed=amed, aslo=aslo, pow_=pow_)

    pixels    = _make_pixels(med_period, slow_period)
    duration_s = int(duration_min * 60)
    t_arr      = np.linspace(0, duration_s, duration_s * 4)

    fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=True)
    fig.suptitle(
        f"pow={pow_}  α={smooth}  dc={dc}  amed={amed}  aslo={aslo}  "
        f"med={med_period:.0f}s  slow={slow_period:.0f}s"
        + ("  [no texture]" if no_texture else ""),
        fontsize=10
    )

    for i, (ax, params) in enumerate(zip(axes, pixels)):
        br_raw = np.array([_brightness(t, *params, **wave_kw) for t in t_arr])
        br_sm  = _ewma(br_raw, smooth)
        c_arr  = [_contributions(t, *params, **wave_kw) for t in t_arr]
        fast   = np.array([c[0] for c in c_arr])
        med    = np.array([c[1] for c in c_arr])
        slow   = np.array([c[2] for c in c_arr])

        ax.fill_between(t_arr / 60, 0, np.clip(fast, 0, None),
                        alpha=0.25, color=_PX_COLORS[i], label="fast texture")
        ax.fill_between(t_arr / 60, np.clip(fast, 0, None),
                        np.clip(fast + med, 0, None),
                        alpha=0.30, color="steelblue", label="medium swell")
        ax.fill_between(t_arr / 60, np.clip(fast + med, 0, None),
                        np.clip(fast + med + slow, 0, None),
                        alpha=0.35, color="gold", label="slow swell")

        ax.plot(t_arr / 60, br_raw, color=_PX_COLORS[i], lw=0.6, alpha=0.3,
                label="raw" if i == 0 else None)
        ax.plot(t_arr / 60, br_sm, color=_PX_COLORS[i], lw=1.4, alpha=0.95,
                label=f"smoothed (α={smooth})" if i == 0 else None)

        for y, ls, alpha, color, label in _REF_LINES:
            ax.axhline(y, ls=ls, color=color, alpha=alpha, lw=1.0,
                       label=label if i == 0 else None)

        ax.set_ylim(-0.02, 1.08)
        ax.set_ylabel(f"Pixel {i}", fontsize=10)
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.1))
        ax.grid(True, which="major", alpha=0.15)
        ax.tick_params(labelsize=8)

    x_step = max(1, int(duration_min / 10))
    axes[-1].set_xlabel("Time (minutes)", fontsize=10)
    axes[-1].xaxis.set_major_locator(ticker.MultipleLocator(x_step))
    axes[-1].xaxis.set_minor_locator(ticker.MultipleLocator(x_step / 4))
    axes[0].legend(loc="upper right", fontsize=8, framealpha=0.85, ncol=5)

    plt.tight_layout()
    plt.show()


interact(
    _plot,
    smooth      = FloatSlider(value=0.30, min=0.05, max=1.0,  step=0.05, description="smooth α",      continuous_update=False),
    pow_        = FloatSlider(value=6.0,  min=1.0,  max=10.0, step=0.5,  description="swell pow",     continuous_update=False),
    amed        = FloatSlider(value=0.56, min=0.10, max=0.90, step=0.02, description="med amp",        continuous_update=False),
    aslo        = FloatSlider(value=0.86, min=0.10, max=0.95, step=0.02, description="slow amp",       continuous_update=False),
    dc          = FloatSlider(value=0.04, min=0.01, max=0.15, step=0.01, description="dc floor",       continuous_update=False),
    med_period  = FloatSlider(value=67.0, min=20.0, max=180.0,step=5.0,  description="med period (s)", continuous_update=False),
    slow_period = FloatSlider(value=331.0,min=120.0,max=600.0,step=10.0, description="slow period (s)",continuous_update=False),
    no_texture  = Checkbox(value=False, description="no texture"),
    duration_min= IntSlider(value=20,   min=5,    max=60,    step=5,    description="duration (min)", continuous_update=False),
);

In [ ]:
# ── Export ────────────────────────────────────────────────────────────────────
# Set these to your final tuned values, then run the cell.
# Copy the printed output into lights.py.

FINAL = dict(
    smooth      = 0.30,
    pow_        = 6,
    amed        = 0.56,
    aslo        = 0.86,
    dc          = 0.04,
    med_period  = 67.0,
    slow_period = 331.0,
)

pixels = _make_pixels(FINAL['med_period'], FINAL['slow_period'])

print("# ── Paste into lights.py ────────────────────────────────────────────")
print(f"_IDLE_DC   = {FINAL['dc']}")
print(f"_IDLE_AMED = {FINAL['amed']}   # medium swell amplitude")
print(f"_IDLE_ASLO = {FINAL['aslo']}   # slow swell amplitude")
print(f"_IDLE_POW  = {FINAL['pow_']}      # swell sharpness")
print()
print("_IDLE_PX = (")
labels = ["pixel 0", "pixel 1", "pixel 2"]
for label, (phi, T1, T2, T3, Tm, t_off_m, Ts, t_off_s) in zip(labels, pixels):
    print(f"    ({phi}, {T1}, {T2}, {T3},  {Tm:.1f}, {t_off_m:.1f},  {Ts:.1f}, {t_off_s:.1f}),   # {label}")
print(")")
print()
print(f"# In IdleSparkle class:")
print(f"_SMOOTH = {FINAL['smooth']}")